# Summarize ET by Crop Type


This notebook provides a step-by-step guide on utilizing the OpenET API's */geodatabase/metadata/ids* endpoint to retrieve field IDs contained within a specified region. It then queries the */geodatabase/timeseries* endpoint to obtain monthly evapotranspiration and precipitation data for all fields within that region. The workflow combines the API results with field boundaries and 2024 Cropland Data Layer (CDL) crop classifications, calculates monthly net evapotranspiration (ET), and summarizes the net ET by crop type.

The first code cell imports the geospatial, tabular-data, plotting, HTTP-request, compression, and CDL lookup packages used throughout the notebook.

## To complete this notebook you will need:
1. [OpenET Account](https://account.etdata.org/login)
2. [OpenET API Key](https://account.etdata.org/settings/api)
3. A valid GeoJSON file. This demo uses the HUC-12 watershed boundary of the Little Shasta River.


To apply this same workflow for another region:
* Convert a shapefile to GeoJSON using **[MapShaper](https://mapshaper.org/)**
* Evaluate GeoJSON compatibility with OpenET data services using **[GeoJSONLint](https://geojsonlint.com/)**





In [ ]:
#@title Import Python dependencies
import calendar
import getpass
import gzip
import json
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import requests
import shapely
from cdl_lookup import cdl_lookup


# Configure your API Key to access OpenET data

#### Go to your [OpenET API account](https://auth.etdata.org/dashboard) page and copy the API Key.

#### Then paste your API Key below after running this cell

The code cell below stores the key in `YOUR_API_KEY` and builds an authorization header for subsequent OpenET requests. Replace the empty string with your key before running the remaining API cells.


In [ ]:

YOUR_API_KEY = "PASTE_KEY_HERE"

header = {"Authorization": YOUR_API_KEY}

### Load boundary as GeoJSON 

* In the geospatial data folder, you can right-click a file and select the option 'Copy path', which you can paste to assign a new `geojson_path`.
* Read the GeoJSON into a GeoDataFrame and derive a region name from the file name.
* Extract the boundary coordinates with `shapely.get_coordinates` and flatten the latitude/longitude pairs into the list format required by the API.

The code cell below performs these steps and prints the region name so you can confirm which boundary is being analyzed.

In [ ]:

geojson_path = "geospatial_data/James_GSA.geojson" 
region_name = geojson_path.split('/')[-1].split('.')[0]
print(region_name)

geojson = gpd.read_file(geojson_path)
geometry = shapely.get_coordinates(geojson.geometry).flatten().tolist()

## Retrieve Field IDs contained within GeoJSON

The following code cells use the region geometry to build the field-level dataset:

* The first request sends the boundary to `/geodatabase/metadata/ids` and decompresses the response into a list of matching field IDs.
* The second request sends those IDs to `/geodatabase/metadata/properties`, then stores the returned attributes—including hectares and the 2024 crop code—in a pandas DataFrame.
* The third request retrieves the corresponding field boundaries from `/geodatabase/metadata/boundaries`.
* The next cell writes those boundaries to a region-specific GeoJSON file and reads them back as a GeoDataFrame.
* The following cell maps CDL crop codes to crop names and joins the crop and area attributes to the field geometries.
* The final cell requests monthly ensemble ET (`ET`) and GridMET precipitation (`Pr`) for the 2024 water year, from October 1, 2023 through September 30, 2024.

Each API response is gzip-compressed, so the code decompresses and decodes it before converting the results into Python objects.

In [ ]:
# This provides the API with your API key.
api_header = {"AUTHORIZATION": YOUR_API_KEY}
# This provides the parameters for the request.
api_query = {
    "geometry": geometry
}

# This is the request itself.
api_request = requests.post(
    url="https://openet-api.org/geodatabase/metadata/ids",
    headers=api_header,
    json=api_query
)

# Parse the response since it is compressed.
fields = eval(gzip.decompress(api_request.content).decode())

In [ ]:
# Retrieve metadata associated with the list of fields

# endpoint arguments
args = {
  "field_ids":fields
}

# query the api
resp = requests.post(
    headers=api_header,
    json=args,
    url="https://openet-api.org/geodatabase/metadata/properties"
)

# unzip the data
meta_data = eval(gzip.decompress(resp.content).decode())
meta_data_df = pd.DataFrame(meta_data)

In [ ]:
# Retrieve field geometries
geo_query = {
    "field_ids": fields,
}

geo_resp = requests.post(
    url="https://openet-api.org/geodatabase/metadata/boundaries",
    headers=api_header,
    json=geo_query
)

geometries = eval(gzip.decompress(geo_resp.content).decode())

In [ ]:

with open(f"geospatial_data/{region_name}_field_boundaries.geojson", "w") as file:
    json.dump(geometries, file, indent=4)

fields_gdf = gpd.read_file(f"geospatial_data/{region_name}_field_boundaries.geojson")



In [ ]:
# Add CDL crop names
meta_data_df["crop_2024_name"] = meta_data_df["crop_2024"].map(cdl_lookup)
# Join crop data to the GeoDataFrame
fields_crop_gdf = fields_gdf.merge(
    meta_data_df[['field_id','hectares','crop_2024','crop_2024_name']],
    on="field_id",
    how="left",
    validate="one_to_one"
)


In [ ]:
#@title Retrieve monthly timeseries data for 2024 water year

api_query = {
    "field_ids": fields,
    "date_range": ["2023-10-01", "2024-09-30"],
    "interval": "monthly",
    "variables": ["et", "pr"],
    "models": ["ensemble"],
    "file_format": "JSON"
}

api_request = requests.post(
    url="https://openet-api.org/geodatabase/timeseries",
    headers=api_header,
    json=api_query
)

timeseries = eval(gzip.decompress(api_request.content).decode())

## Combine all the data together

The code cell below converts the time-series response into a pandas DataFrame, separates ET and precipitation records, and joins them by field and month. It converts both variables from millimeters to inches, adds field area in acres, and calculates monthly net ET (`ETn`) as `max(0, ET - precipitation)`. This is a simple water-balance estimate that assumes precipitation is 100% effective during the growing season and that there is little or no soil-water carryover from winter to spring. Later examples will show how a new water balance model can better represent soil water carryover.

Finally, the code extracts month and year values, averages `ETn` by field and month, pivots months into columns, and joins those monthly values to the field GeoDataFrame. The resulting `merged_gdf` contains field geometry, crop classification, area, and monthly net-ET values for the analyses below.

In [ ]:

df_meta =pd.DataFrame.from_records(meta_data)[["field_id","hectares"]]
df_meta["acres"] = df_meta["hectares"]*0.404686
df = pd.DataFrame.from_records(timeseries)
df["time"] = pd.to_datetime(df["time"])

# separate et and precipitation then merge dataframes
et_df = df.query("collection == 'ensemble_et'")
et_df = et_df.drop(columns="collection").rename(columns={"value_mm": "et"})

pr_df = df.query("collection == 'gridmet_pr'")
pr_df = pr_df.drop(columns="collection").rename(columns={"value_mm": "pr"})

df = pd.merge(et_df, pr_df, how="inner", on=["field_id", "time"])
cols_to_convert = ["et", "pr"]

# Convert units from mm to inches
df[cols_to_convert] = df[cols_to_convert] / 25.4

df_et_pr = pd.merge(df, df_meta[["field_id","acres"]], how="inner", on=["field_id"])

# Calculate monthly ETn by max(0, ET-Pr) following:
# While there are more advanced approaches. This method assumes:
# * 100% of precipitation is during growing season
# * minimal to 0 soil water carry over
def calculate_etn(df):
    return max(0, df["et"] - df["pr"])

df_et_pr["etn"] = df_et_pr.apply(calculate_etn, axis=1)
# Add colums month and year to aggregate data
df_et_pr['month']=df_et_pr.time.dt.month
df_et_pr['year']=df_et_pr.time.dt.year

df_field_months = df_et_pr.groupby(['field_id','month'])[['etn']].mean().reset_index()
# Pivot data to get columns of months and rows of fields
pivot_df = df_field_months.pivot(index="field_id", columns="month", values="etn")
merged_gdf = fields_crop_gdf.merge(pivot_df.reset_index(), on='field_id', how='left')


## Analyze ET by crop

### Show Crop Types for the Region in 2024 to Select a Crop to Analyze

The next code cell lists each unique 2024 CDL crop name in the region and counts how many fields are assigned to that crop. Use this list of crops to choose a crop to evaluate water demands.

In [ ]:
print(f'Unique crops in {region_name}\n\n')
for crop in merged_gdf['crop_2024_name'].unique():
    print(crop, f'\n\t\t\t\t\ttotal {crop} fields: '+str(merged_gdf.loc[merged_gdf['crop_2024_name']== crop].crop_2024_name.count()))

### Select a Crop Type to Analyze

Enter one of the crop names printed above in `selected_crop`. The following cells then filter the field dataset to that crop, calculate seasonal mean net ET for the configured MAM and JJA seasons, and produce maps using a shared color scale. Selected-crop fields are outlined in black so they can be compared with all fields in the region.

The final plotting cell orders the water-year months from October through September, draws one light-gray line for each selected field, and overlays a bold crop-level mean line. The chart reports monthly `ET - Pr` in inches and includes the number of selected fields.

In [ ]:


selected_crop = "Alfalfa"


### Make a map of all fields within the region. Highlight fields using a black border for fields with the seleted crop.

In [ ]:

merged_gdf = fields_crop_gdf.merge(pivot_df.reset_index(), on='field_id', how='left')
selected_fields = merged_gdf[
    merged_gdf["crop_2024_name"].eq(selected_crop)
]
# Define seasons
seasons = {
    # 'DJF': [12, 1, 2],  # December, January, February
    'MAM': [3, 4, 5],   # March, April, May
    'JJA': [6, 7, 8],   # June, July, August
    # 'SON': [9, 10, 11]  # September, October, November
}

# Calculate seasonal means and store in new columns
for season_name, months in seasons.items():
    # Handle DJF by considering month 12 as part of the winter season starting in the previous year
    # For simplicity, we are taking the mean of the specified months from the pivot_df
    # Assuming the pivot_df already contains the 'etn' values for these months
    merged_gdf[f'etn_{season_name}'] = merged_gdf[months].mean(axis=1)


# Plotting by season
fig, axes = plt.subplots(1, 2, figsize=(15, 10), constrained_layout=True)
axes = axes.flatten()

# Determine common vmin and vmax for all plots
all_etn_values = []
for season_name, _ in seasons.items():
    all_etn_values.extend(merged_gdf[f'etn_{season_name}'].dropna().tolist())

vmin = min(all_etn_values) if all_etn_values else 0
vmax = max(all_etn_values) if all_etn_values else 1 # Avoid division by zero if all_etn_values is empty

for i, (season_name, months) in enumerate(seasons.items()):
    ax = axes[i]
    merged_gdf.plot(
        column=f'etn_{season_name}',
        cmap='terrain_r',
        linewidth=0.8,
        ax=ax,
        edgecolor='0.8',
        legend=False, # Set to False to prevent individual colorbars
        vmin=vmin, # Apply common vmin
        vmax=vmax # Apply common vmax
    )
    selected_fields.plot(
        ax=ax,
        facecolor="none",
        edgecolor="k",
        linewidth=2.,
        zorder=2
    )

    ax.set_title(
        f"ETnet for {season_name}\n"
    )
    ax.set_axis_off()

# # Add a single colorbar
# cbar = fig.colorbar(cmap='mako_r', plt.cm.ScalarMappable(cmap='', norm=plt.Normalize(vmin=vmin, vmax=vmax)), ax=axes.ravel().tolist(),shrink=0.6, pad=0.05)
cbar = fig.colorbar(plt.cm.ScalarMappable(cmap='terrain_r', norm=plt.Normalize(vmin=vmin, vmax=vmax)), ax=axes.ravel().tolist(), shrink=0.6, pad=0.05)
# cbar.set_label('Seasonal ETnet (inches)')
plt.suptitle('Seasonal ETnet by Field ID' + f"Highlighted: {selected_crop} "
        f"({len(selected_fields)} fields)", fontsize=16, y=0.9)
plt.show()

### Plot annual time series of ETn for fields of the specified crop

In [ ]:



crop_plot_df = merged_gdf.loc[
    merged_gdf["crop_2024_name"] == selected_crop
].copy()

# Plot October through September
month_order = [10, 11, 12, 1, 2, 3, 4, 5, 6, 7, 8, 9]

# Extract monthly values
monthly_data = crop_plot_df[month_order]

# Sequential x-axis positions
x = np.arange(len(month_order))

fig, ax = plt.subplots(figsize=(10, 6))

# Plot each field as a thin, light-gray line
for _, field in monthly_data.iterrows():
    ax.plot(
        x,
        field.values,
        color="lightgray",
        linewidth=0.6,
        alpha=0.6,
        zorder=1
    )

# Plot the crop-level mean as a bold line
mean_values = monthly_data.mean(axis=0).values

ax.plot(
    x,
    mean_values,
    color="black",
    linewidth=3,
    marker="o",
    label="Mean ET - Pr",
    zorder=2
)

# Format the x-axis
month_labels = [calendar.month_abbr[m] for m in month_order]

ax.set_xticks(x)
ax.set_xticklabels(month_labels)

ax.set_xlabel("Month")
ax.set_ylabel("ET - Pr (inches)")
ax.set_title(
    f"$ET_{{net}}$ by Field and Month for {selected_crop}\n"
    f"Number of fields: {len(crop_plot_df)}"
)

ax.grid(axis="y", alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()